# Winning Jeopardy!

"Jeopardy!" is a popular TV quiz show in the US where participants answer questions to win money. It has been on the air since 1964 and has grown to be a major force in US Popular Culture.

## Aim

The goal of this project is to analyze a dataset of past Jeopardy! questions to gain insights into any patterns that may exist in an effort to aid contestants in preparing for an appearance on the show. 

The dataset was acquired from the Subreddit [r/datasets](https://www.reddit.com/r/datasets/comments/1uyd0t/200000_jeopardy_questions_in_a_json_file/), comprising over 216,930 Jeopardy questions from shows aired between 1984 and 2012. The original data was scraped from [J-Archive](https://www.j-archive.com/index.php) a fan-made archive of Jeopardy! games.

## Exploring and Cleaning the Dataset

In [90]:
import pandas as pd
import numpy as np

pd.options.display.max_colwidth = None
pd.options.display.max_columns = 30

jeopardy = pd.read_csv('JEOPARDY_CSV.csv')

jeopardy.head()

,Show Number,Air Date,Round,Category,Value,Question,Answer
0,4680,2004-12-31,Jeopardy!,HISTORY,$200,"For the last 8 years of his life, Galileo was under house arrest for espousing this man's theory",Copernicus
1,4680,2004-12-31,Jeopardy!,ESPN's TOP 10 ALL-TIME ATHLETES,$200,"No. 2: 1912 Olympian; football star at Carlisle Indian School; 6 MLB seasons with the Reds, Giants & Braves",Jim Thorpe
2,4680,2004-12-31,Jeopardy!,EVERYBODY TALKS ABOUT IT...,$200,"The city of Yuma in this state has a record average of 4,055 hours of sunshine each year",Arizona
3,4680,2004-12-31,Jeopardy!,THE COMPANY LINE,$200,"In 1963, live on ""The Art Linkletter Show"", this company served its billionth burger",McDonald's
4,4680,2004-12-31,Jeopardy!,EPITAPHS & TRIBUTES,$200,"Signer of the Dec. of Indep., framer of the Constitution of Mass., second President of the United States",John Adams


In [91]:
jeopardy.columns

Index(['Show Number', ' Air Date', ' Round', ' Category', ' Value',
       ' Question', ' Answer'],
      dtype='str')

Some column names containg trailing whitespace. This can be removed to ensure consistent column names.

In [92]:
jeopardy.columns = [col.strip() for col in jeopardy.columns]
jeopardy.columns

Index(['Show Number', 'Air Date', 'Round', 'Category', 'Value', 'Question',
       'Answer'],
      dtype='str')

In [93]:
jeopardy.info()

<class 'pandas.DataFrame'>
RangeIndex: 216930 entries, 0 to 216929
Data columns (total 7 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   Show Number  216930 non-null  int64
 1   Air Date     216930 non-null  str  
 2   Round        216930 non-null  str  
 3   Category     216930 non-null  str  
 4   Value        213296 non-null  str  
 5   Question     216930 non-null  str  
 6   Answer       216927 non-null  str  
dtypes: int64(1), str(6)
memory usage: 11.6 MB


There seem to be three missing answers, this warrants further investigation.

In [94]:
jeopardy[jeopardy["Answer"].isnull()]

,Show Number,Air Date,Round,Category,Value,Question,Answer
94817,4346,2003-06-23,Jeopardy!,"GOING ""N""SANE",$200,"It often precedes ""and void""",NaN
143297,6177,2011-06-21,Double Jeopardy!,NOTHING,$400,"This word for ""nothing"" precedes ""and void"" to mean ""not valid""",NaN
178922,4573,2004-06-23,Jeopardy!,MUCH ADO ABOUT NOTHING,$200,"Completes the title of the 1939 book by Agatha Christie ""And Then There Were...""",NaN


Funnily, the missing answers are "Null" and "None" - pandas has set these to `np.nan` values by default. This can be rectified by simply reassigning the correct answers as strings.

In [95]:
jeopardy.iloc[94817,6] = "Null"
jeopardy.iloc[143297,6] = "Null"
jeopardy.iloc[178922,6] = "None"

jeopardy.info()

<class 'pandas.DataFrame'>
RangeIndex: 216930 entries, 0 to 216929
Data columns (total 7 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   Show Number  216930 non-null  int64
 1   Air Date     216930 non-null  str  
 2   Round        216930 non-null  str  
 3   Category     216930 non-null  str  
 4   Value        213296 non-null  str  
 5   Question     216930 non-null  str  
 6   Answer       216930 non-null  str  
dtypes: int64(1), str(6)
memory usage: 11.6 MB


We also observe a number of missing values in the `Value` field cells. Looking deeper, all of these rows correspond to questions asked in the **final round** of the show. This seems to be the standard format for the show - final round questions are not worth any prize money.

In [96]:
jeopardy[jeopardy["Value"].isnull()].sample(10)

,Show Number,Air Date,Round,Category,Value,Question,Answer
11123,4300,2003-04-18,Final Jeopardy!,ORGANIZATIONS,NaN,"""Climb the mountains and get their good tidings"" was a goal of this group at its 19th century founding",Sierra Club
47456,3268,1998-11-18,Final Jeopardy!,20th CENTURY NOVELS,NaN,"With the same initials as the author, Harry Haller is the loner protagonist of this 1927 German novel",Steppenwolf (by Hermann Hesse)
97538,3649,2000-06-15,Final Jeopardy!,THE SUPREME COURT,NaN,These 2 justices who graduated at the top of their classes were both first offered jobs as typists by the top law firms,Ruth Bader Ginsburg & Sandra Day O'Connor
26845,5481,2008-06-09,Final Jeopardy!,THE INTERNET,NaN,"On March 10, 2003 this nation got control of the .af Internet domain",Afghanistan
185423,5316,2007-10-22,Final Jeopardy!,QUOTATIONS FROM B.C.,NaN,"This work says, ""Victorious warriors win first & then go to war, while defeated warriors go to war first & then seek to win""",The Art of War (by Sun Tzu)
184291,3584,2000-03-16,Final Jeopardy!,SPACE,NaN,"On Nov. 13, 1999 a body circling HD 209458 became the first new planet to be photographed since this one",Pluto
196179,5003,2006-05-17,Final Jeopardy!,BRITISH MONARCHS,NaN,The last British monarch who was not the child of a monarch,Queen Victoria
95344,6137,2011-04-26,Final Jeopardy!,PLAYWRIGHTS,NaN,"This Brit won Tonys for Best Play in 1968, 1976, 1984 & 2007; in the '90s he settled for the 1998 Best Screenplay Oscar",Tom Stoppard
61522,5022,2006-06-13,Final Jeopardy!,LITERARY QUOTES,NaN,"""I would like to take the great DiMaggio fishing"" is a line from this 1952 work; like DiMaggio, it's an American classic",The Old Man and the Sea (by Ernest Hemingway)
18259,2853,1997-01-15,Final Jeopardy!,FAMOUS NAMES,NaN,"Before achieving fame in Hollywood, he was a cosmetician to the Russian royal court",Max Factor


### Standardizing Question and Answer Fields

It is essential that punctuation is removed and the case of each letter in the `Question` and `Answer` fields is standardized such that all instances of a word are treated equally (for example, "The" and "the" should be recognized as the same word).

The `Value` column must also be cleaned by converting the value to an integer and setting null values to 0.

Cleaning functions can be applied to each column to achieve this.

In [97]:
import re

def clean_text(text):
    
    text = str(text).lower()                # Convert to string and lowercase
    text = re.sub(r"[^\w\s]", "", text)     # Replace non-alphanumeric characters and whitespace with nothing - this removes punctuation
    text = re.sub(r"\s+", " ", text)        # Replace multiple spaces with single space
    return text

def clean_value(value):
    
    value = str(value)
    value = re.sub(r"[$]", "", value) # Remove dollar sign
    
    try:
        value = int(value)            # Convert to integer
        
    except Exception:
        value = 0                     # Handles non-numeric values like NaN, None, setting them to 0
        
    return value
  
jeopardy["Clean Question"] = jeopardy["Question"].apply(clean_text)

jeopardy["Clean Answer"] = jeopardy["Answer"].apply(clean_text)

jeopardy["Clean Value"] = jeopardy["Value"].apply(clean_value)

jeopardy["Air Date"] = pd.to_datetime(jeopardy["Air Date"])  # Convert to datetime

jeopardy.head()

,Show Number,Air Date,Round,Category,Value,Question,Answer,Clean Question,Clean Answer,Clean Value
0,4680,2004-12-31,Jeopardy!,HISTORY,$200,"For the last 8 years of his life, Galileo was under house arrest for espousing this man's theory",Copernicus,for the last 8 years of his life galileo was under house arrest for espousing this mans theory,copernicus,200
1,4680,2004-12-31,Jeopardy!,ESPN's TOP 10 ALL-TIME ATHLETES,$200,"No. 2: 1912 Olympian; football star at Carlisle Indian School; 6 MLB seasons with the Reds, Giants & Braves",Jim Thorpe,no 2 1912 olympian football star at carlisle indian school 6 mlb seasons with the reds giants braves,jim thorpe,200
2,4680,2004-12-31,Jeopardy!,EVERYBODY TALKS ABOUT IT...,$200,"The city of Yuma in this state has a record average of 4,055 hours of sunshine each year",Arizona,the city of yuma in this state has a record average of 4055 hours of sunshine each year,arizona,200
3,4680,2004-12-31,Jeopardy!,THE COMPANY LINE,$200,"In 1963, live on ""The Art Linkletter Show"", this company served its billionth burger",McDonald's,in 1963 live on the art linkletter show this company served its billionth burger,mcdonalds,200
4,4680,2004-12-31,Jeopardy!,EPITAPHS & TRIBUTES,$200,"Signer of the Dec. of Indep., framer of the Constitution of Mass., second President of the United States",John Adams,signer of the dec of indep framer of the constitution of mass second president of the united states,john adams,200


In [98]:
jeopardy.info()

<class 'pandas.DataFrame'>
RangeIndex: 216930 entries, 0 to 216929
Data columns (total 10 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   Show Number     216930 non-null  int64         
 1   Air Date        216930 non-null  datetime64[us]
 2   Round           216930 non-null  str           
 3   Category        216930 non-null  str           
 4   Value           213296 non-null  str           
 5   Question        216930 non-null  str           
 6   Answer          216930 non-null  str           
 7   Clean Question  216930 non-null  str           
 8   Clean Answer    216930 non-null  str           
 9   Clean Value     216930 non-null  int64         
dtypes: datetime64[us](1), int64(2), str(7)
memory usage: 16.6 MB


## How Often do Answers Appear in Questions?

In [99]:
def answer_words_in_question(row):
  
  """Returns the proportion of words in the answer that are also present in the question."""
  
  question = row["Clean Question"]
  answer = row["Clean Answer"]
  
  split_question = question.split(" ")
  split_answer = answer.split(" ")
  
  match_count = 0
  
  if "the" in split_answer:          # Remove "the" and "a" from answer words as these are common to both questions and answers, but do not have use in answering our analysis question.
    split_answer.remove("the")
    
  if "a" in split_answer:
    split_answer.remove("a")
    
  if len(split_answer) == 0:
    return 0
  
  for word in split_answer:
    if word in split_question:
      match_count += 1
    
  return match_count / len(split_answer) 

jeopardy["Answer in Question"] = jeopardy.apply(answer_words_in_question, axis=1)

jeopardy.head()

,Show Number,Air Date,Round,Category,Value,Question,Answer,Clean Question,Clean Answer,Clean Value,Answer in Question
0,4680,2004-12-31,Jeopardy!,HISTORY,$200,"For the last 8 years of his life, Galileo was under house arrest for espousing this man's theory",Copernicus,for the last 8 years of his life galileo was under house arrest for espousing this mans theory,copernicus,200,0.0
1,4680,2004-12-31,Jeopardy!,ESPN's TOP 10 ALL-TIME ATHLETES,$200,"No. 2: 1912 Olympian; football star at Carlisle Indian School; 6 MLB seasons with the Reds, Giants & Braves",Jim Thorpe,no 2 1912 olympian football star at carlisle indian school 6 mlb seasons with the reds giants braves,jim thorpe,200,0.0
2,4680,2004-12-31,Jeopardy!,EVERYBODY TALKS ABOUT IT...,$200,"The city of Yuma in this state has a record average of 4,055 hours of sunshine each year",Arizona,the city of yuma in this state has a record average of 4055 hours of sunshine each year,arizona,200,0.0
3,4680,2004-12-31,Jeopardy!,THE COMPANY LINE,$200,"In 1963, live on ""The Art Linkletter Show"", this company served its billionth burger",McDonald's,in 1963 live on the art linkletter show this company served its billionth burger,mcdonalds,200,0.0
4,4680,2004-12-31,Jeopardy!,EPITAPHS & TRIBUTES,$200,"Signer of the Dec. of Indep., framer of the Constitution of Mass., second President of the United States",John Adams,signer of the dec of indep framer of the constitution of mass second president of the united states,john adams,200,0.0


In [100]:
answer_in_question_counts = jeopardy["Answer in Question"].value_counts()

answer_in_question_counts

Answer in Question
0.000000    196666
0.500000     10780
0.333333      4051
0.250000      1461
1.000000      1414
0.666667       760
0.200000       608
0.400000       280
0.166667       251
0.142857       116
0.750000       108
0.285714        71
0.600000        70
0.125000        57
0.428571        29
0.222222        27
0.375000        26
0.800000        25
0.111111        21
0.571429        19
0.300000         8
0.714286         7
0.833333         7
0.100000         7
0.181818         6
0.272727         5
0.083333         5
0.625000         5
0.857143         4
0.153846         4
0.230769         4
0.777778         2
0.444444         2
0.583333         2
0.555556         2
0.545455         2
0.363636         2
0.090909         2
0.350000         1
0.636364         1
0.700000         1
0.307692         1
0.368421         1
0.071429         1
0.133333         1
0.117647         1
0.818182         1
0.454545         1
0.066667         1
0.266667         1
0.058824         1
0.384615    

The majority of answers do not contain any words featured in the question. 

In [101]:
number_of_questions_containing_words_in_answer = np.sum([count for index, count in enumerate(answer_in_question_counts) if index > 0])

print(f"""The number of questions which contain words in the answer: {number_of_questions_containing_words_in_answer:,}
      
This makes up {number_of_questions_containing_words_in_answer / len(jeopardy) * 100:.2f}% of the total questions.""")

The number of questions which contain words in the answer: 20,264

This makes up 9.34% of the total questions.


The fact that only around 9% of the answers in the dataset have words featured in the question means that this avenue of analysis should certainly not be the focus for preparing a contestant to win the show. This does not however mean that there is no merit in looking into this further.

Below, insights are uncovered for questions that do have answers that are partly or fully featured in the question.

In [102]:
jeopardy[jeopardy["Answer in Question"] > 0].sample(10)

,Show Number,Air Date,Round,Category,Value,Question,Answer,Clean Question,Clean Answer,Clean Value,Answer in Question
48238,3692,2000-09-26,Jeopardy!,ROYALTY,$200,"This ""Great"" czar got rid of his powerful half-sister Sophia by sending her off to a convent",Peter the Great,this great czar got rid of his powerful halfsister sophia by sending her off to a convent,peter the great,200,0.500000
152070,1651,1991-11-04,Double Jeopardy!,AUTHORS FROM GEORGIA,$400,Poet & novelist James Dickey was once poetry consultant to this national library,Library of Congress,poet novelist james dickey was once poetry consultant to this national library,library of congress,400,0.333333
182877,2839,1996-12-26,Jeopardy!,THE EARLY 1900S,$600,"This age of fashion was ushered into England January 22, 1901",The Edwardian Age,this age of fashion was ushered into england january 22 1901,the edwardian age,600,0.500000
17703,5770,2009-10-16,Double Jeopardy!,GETTING CONFRONTATIONAL,$400,Protestant & Catholic disagreement about the 1555 Peace of Augsburg was a cause of this numeric war,the Thirty Years' War,protestant catholic disagreement about the 1555 peace of augsburg was a cause of this numeric war,the thirty years war,400,0.333333
36240,5430,2008-03-28,Double Jeopardy!,I'M FEELING BOOKISH,$400,"A phrase in Shakespeare's ""Timon of Athens"" became the title of this Capote crime book",In Cold Blood,a phrase in shakespeares timon of athens became the title of this capote crime book,in cold blood,400,0.333333
60347,6208,2011-09-21,Jeopardy!,OUR NEW COMPUTER OVERLORDS,$800,"Thinking Machines Corp. pioneered this processing --as the geometric name indicates, many processors work side by side",parallel processing,thinking machines corp pioneered this processing as the geometric name indicates many processors work side by side,parallel processing,800,0.500000
81796,4295,2003-04-11,Double Jeopardy!,A VERY BIZET MAN,$1200,In 1875 Bizet was honored by being a chevalier of this French legion,the Legion of Honor,in 1875 bizet was honored by being a chevalier of this french legion,the legion of honor,1200,0.666667
61549,3048,1997-11-26,Jeopardy!,"SOMETHING'S ""FISH""Y",$500,"It means to do one thing or another, but stop stalling",Fish or cut bait,it means to do one thing or another but stop stalling,fish or cut bait,500,0.250000
3372,5334,2007-11-15,Jeopardy!,"""IRA""",$1000,This type of medieval play often shows the Virgin Mary coming to the rescue,a miracle play,this type of medieval play often shows the virgin mary coming to the rescue,a miracle play,1000,0.500000
82474,3824,2001-03-29,Double Jeopardy!,BELOW THE WAIST,$1000,The major pressure point for leg injuries is where this artery crosses the joint between the pelvis & leg,Femoral artery,the major pressure point for leg injuries is where this artery crosses the joint between the pelvis leg,femoral artery,1000,0.500000


Looking at a sample of answers containing some proportion of their words in the question, it does not seem to be particularly helpful in devising a strategy. In many cases, it is already given in the question that the answer will contain some words from the question. 

For instance: "French film star Simone Signoret married 2 men named Yves: director Yves Allegret & this actor" - we can simply deduce the answer will be two words long "Yves ___" without any prior knowledge.

It is an interesting observation however to look into the `HIDDEN COUNTRIES` category. This appears to be a style of question that features a country name hidden within the question statement. An example from the dataset: "There are many who talk in dialect in this country" - the answer is the hidden country name **India**. It would therefore be wise to become familiar with country names and practice this style of question.

In [103]:
jeopardy[jeopardy["Category"] == "HIDDEN COUNTRIES"].sample(5)

,Show Number,Air Date,Round,Category,Value,Question,Answer,Clean Question,Clean Answer,Clean Value,Answer in Question
82676,4914,2006-01-12,Jeopardy!,HIDDEN COUNTRIES,$600,Hype rubber chickens,Peru,hype rubber chickens,peru,600,0.0
130264,5883,2010-03-24,Jeopardy!,HIDDEN COUNTRIES,$400,"Truly, you have created a beauteous painting",Spain (in beauteous painting),truly you have created a beauteous painting,spain in beauteous painting,400,0.5
130270,5883,2010-03-24,Jeopardy!,HIDDEN COUNTRIES,$600,Your job is to find the hidden marksman,Denmark (in hidden marksman),your job is to find the hidden marksman,denmark in hidden marksman,600,0.5
82682,4914,2006-01-12,Jeopardy!,HIDDEN COUNTRIES,$800,I've been through analysis,Ghana,ive been through analysis,ghana,800,0.0
130275,5883,2010-03-24,Jeopardy!,HIDDEN COUNTRIES,$800,Beach walking may require you to put on galoshes,Tonga (in put on galoshes),beach walking may require you to put on galoshes,tonga in put on galoshes,800,0.6


Above, we also found 1,414 of the answers in the dataset are fully contained within the question. It could be worth looking into this to aid in preparation for winning the show.

In [104]:
jeopardy[jeopardy["Answer in Question"] == 1].sample(10)

,Show Number,Air Date,Round,Category,Value,Question,Answer,Clean Question,Clean Answer,Clean Value,Answer in Question
123713,6033,2010-12-01,Jeopardy!,STUPID ANSWERS,$200,"It was the phase of the moon on Nov. 16, 2009, the date the movie ""New Moon"" premiered",the new moon,it was the phase of the moon on nov 16 2009 the date the movie new moon premiered,the new moon,200,1.0
193416,4702,2005-02-01,Jeopardy!,THE LARGEST IN AREA,$1000,"Japan, Jamaica, Jordan",Japan,japan jamaica jordan,japan,1000,1.0
137223,5960,2010-07-09,Jeopardy!,HUMAN BODY NUMBERS,$600,"Of 106, 206 or 506, the approximate number of bones in the human body",206,of 106 206 or 506 the approximate number of bones in the human body,206,600,1.0
111996,5836,2010-01-18,Double Jeopardy!,ONE OF THESE KINGS IS NOT LIKE THE OTHERS,$400,"Henry VII, Louis XIV, Richard III",Louis XIV,henry vii louis xiv richard iii,louis xiv,400,1.0
201216,5365,2007-12-28,Jeopardy!,MULTIPLE CHOICE,$600,"Of France, Italy or Russia, the one whose national anthem begins, ""Arise children of the fatherland""",France,of france italy or russia the one whose national anthem begins arise children of the fatherland,france,600,1.0
179468,5105,2006-11-17,Jeopardy!,THE LONG & THE SHORT OF IT,$200,"Of 25,000, 250,000 or 1 million miles, the one closest to the length of the equator","25,000 miles",of 25000 250000 or 1 million miles the one closest to the length of the equator,25000 miles,200,1.0
185547,3078,1998-01-07,Jeopardy!,WET & WILD,$100,"Of a bird, a fish or an insect, what a water boatman is",Insect,of a bird a fish or an insect what a water boatman is,insect,100,1.0
83860,5041,2006-07-10,Jeopardy!,STUPID ANSWERS,$600,In 1501 Venetian printers began using the moveable type of this to print music,moveable type,in 1501 venetian printers began using the moveable type of this to print music,moveable type,600,1.0
89532,5537,2008-10-07,Double Jeopardy!,NAME THE VEEP,$2000,"Levi P. Morton, Samuel L. Miller, George F. McGinnis",(Levi) Morton,levi p morton samuel l miller george f mcginnis,levi morton,2000,1.0
21724,4609,2004-09-23,Jeopardy!,KOALA TIME,$400,"Of 6, 16, or 60 years, the closest to the average life expectancy of a koala",16,of 6 16 or 60 years the closest to the average life expectancy of a koala,16,400,1.0


Some categories such as `THE LARGEST IN AREA` list three countries and the contestant must choose the correct answer. It certainly seems worthwhile to study up on geographical general knowledge for this category (and others such as `NOT A NATIONAL CAPITAL`).

Moving past this, the `STUPID ANSWERS` category looks interesting:

In [105]:
avg = jeopardy[jeopardy["Category"] == "STUPID ANSWERS"]["Answer in Question"].mean()

print(f"""The average proportion of answer words in the question for the category "STUPID ANSWERS": {avg:.2f}""")

jeopardy[jeopardy["Category"] == "STUPID ANSWERS"].sample(5)

The average proportion of answer words in the question for the category "STUPID ANSWERS": 0.66


,Show Number,Air Date,Round,Category,Value,Question,Answer,Clean Question,Clean Answer,Clean Value,Answer in Question
148907,3527,1999-12-28,Jeopardy!,STUPID ANSWERS,$500,"(Hi, I'm Jason Williams of the New Jersey Nets) In 1999 this flashy Sacramento Kings guard was second in voting for NBA Rookie of the Year",Jason Williams,hi im jason williams of the new jersey nets in 1999 this flashy sacramento kings guard was second in voting for nba rookie of the year,jason williams,500,1.0
109645,3653,2000-06-21,Jeopardy!,STUPID ANSWERS,$500,"This great Hindu hero's story is told in the ""Ramayana""",Rama,this great hindu heros story is told in the ramayana,rama,500,0.0
144419,4173,2002-10-23,Jeopardy!,STUPID ANSWERS,$1000,Frankfurt is the main city on this German river,Main,frankfurt is the main city on this german river,main,1000,1.0
123731,6033,2010-12-01,Jeopardy!,STUPID ANSWERS,$800,You can form your own rock band & tour the world with this 2007 video game from MTV games,Rock Band,you can form your own rock band tour the world with this 2007 video game from mtv games,rock band,800,1.0
53869,4637,2004-11-03,Jeopardy!,STUPID ANSWERS,$600,In 2004 publisher David Carey announced that more Californians than New Yorkers subscribed to this magazine,The New Yorker,in 2004 publisher david carey announced that more californians than new yorkers subscribed to this magazine,the new yorker,600,0.5


This category seems to reward quick thinking/instinct over thoughtfulness. Often, the answer to the question is present in the question itself. This seems to be a style of question for contestants to become familiar with and practice.

## Recycled Questions

A second avenue of interest is looking into how often questions are reused across the show. This cannot be fully answered using this dataset of questions, as it does not fully encompass every single question that has ever been asked on the show over its 62 year runtime. Nevertheless, insights can still be garnered from analysis of this sample.

Below, a `Question Overlap` column is defined. 

>What this is attempting to describe is, for a given question, what is the proportion of single word terms (words that are six or more characters long) that have appeared in previous questions asked on the show?

The reasoning behind restricting the words in the question to be six or more characters long is to filter out filler words such as "the" and "than". These are common, but do not provide information about any particular question.

In [106]:
question_overlap = []

terms_previously_used = set()                       # Words that are 6 or more letters long will be stored in the set.

jeopardy = jeopardy.sort_values("Air Date")

for i, row in jeopardy.iterrows():
  split_question = row["Clean Question"].split(" ")
  split_question = [word for word in split_question if len(word) > 5]  # Remove words that are less than 6 letters long
  match_count = 0       
  
  for word in split_question:
    if word in terms_previously_used:
      match_count += 1                              # If word in question is also in terms_previously_used, increment match_count.
    
  for word in split_question:                       # Add words in question to terms_previously_used. This must be placed after the if statement above to avoid over-counting.
    terms_previously_used.add(word) 
     
  if len(split_question) > 0:
    match_count /= len(split_question)
      
  question_overlap.append(match_count)
      
jeopardy["Question Overlap"] = question_overlap

In [107]:
jeopardy["Question Overlap"].mean()

np.float64(0.872176637774269)

This tells us for the average Jeopardy! question, ~87% of words that are six-letters or longer have been used in previous questions. This finding is not very significant since it only considers single word terms. Still, it may mean it is worth looking into this further.

---

Below, we attempt to approximate the proportion of questions that have appeared on the show previously. We assume for a question in a given category, if the answer to this question is identical to the answer of another question that has appeared in the same category, the questions are either the same or very similar.

The code below:

* Iterates through every row in the dataset. For each question, a `category_answer_pair` tuple is created which stores the question's category and answer.
* If the category and answer for the current question in the loop is already contained within the set of previous category-answer pairs, the question will be deemed similar to a previously asked question and the corresponding question, category and answer is appended to the list of `similar_questions`.
* The `similar_questions` list of tuples is then converted to a dataframe and unpacked into category, question and answer columns for further inspection.

In [108]:
similar_questions = []
previous_category_answer_pairs = set()

for i, row in jeopardy.iterrows():
  
  question = row["Question"]
  category = row["Category"]
  answer = row["Clean Answer"]
  category_answer_pair = (category, answer)
  
  if (row["Category"], row["Clean Answer"]) in previous_category_answer_pairs:
    similar_questions.append((category, question, answer))
    
  previous_category_answer_pairs.add(category_answer_pair)

similar_questions = pd.Series(similar_questions).to_frame(name="Category-Question Pair")

similar_questions["Category"] = similar_questions["Category-Question Pair"].apply(lambda x: x[0])
similar_questions["Question"] = similar_questions["Category-Question Pair"].apply(lambda x: x[1])
similar_questions["Answer"] = similar_questions["Category-Question Pair"].apply(lambda x: x[2])

similar_questions.drop("Category-Question Pair", axis=1, inplace=True)

grouped = similar_questions.groupby(["Category", "Question"]).agg({"Answer": "first"})

In [109]:
proportion_of_similar_questions = len(similar_questions) / len(jeopardy)

print(f"The percentage of questions in the sample that are either the same as or similar to another question: {proportion_of_similar_questions * 100:.2f}%")

The percentage of questions in the sample that are either the same as or similar to another question: 4.91%


It has been found that just under 5% of the questions contained in the dataset are the same as or highly similar to question(s) asked previously. This is a fairly significant proportion - we can approximate **just under 1 in 20 questions have been asked previously in some form or another** (assuming the format of the show and style of questions have not changed significantly since this dataset was acquired).

---

In [110]:
grouped.head(30)

Answer
Category           Question                                                                                                                                                                                
"A" IN GEOGRAPHY   It's the capital of Jordan                                                                                                                                                         amman
                   Known to the Romans as Numidia, this large African country borders the Mediterranean Sea                                                                                         algeria
                   This Scottish seaport lies between the rivers Dee & Don                                                                                                                         aberdeen
                   This ancient city is the capital of Greece                                                                                                                                        athens
                   This country controls the eastern half of Tierra del Fuego, largest island in an archipelago of the same name                                                                  argentina
"A" MEN            1994's "Three Tall Women" earned him his third Pulitzer Prize for Drama                                                                                                     edward albee
                   This famed fashion photographer passed away in October 2004                                                                                                               richard avedon
"A" PLUS           Whether his name is Bud or not, he's the head man at a monastery                                                                                                                   abbot
"A.C."             His Third Symphony includes his earlier "Fanfare for the Common Man"                                                                                                       aaron copland
                   The U.S. Navy has 12 of these equipped with steam-driven catapults                                                                                                     aircraft carriers
"AA"               From the Arabic, it's a low bow, or a salutation meaning "peace"                                                                                                                  salaam
"AW", SHUCKS       Developed by sailors to pass the time, <a href="http://www.j-archive.com/media/2011-06-14_J_21.jpg" target="_blank">it</a>'s the art of carving on whalebone or ivory          scrimshaw
"B" IN FASHION     For many women their wardrobe includes a black one of these jackets as well as a double breasted navy one                                                                       a blazer
"B" IN GEOGRAPHY   The NFL Europe's Dragons play their home games in this Spanish city that hosted the 1992 Summer Olympics                                                                       barcelona
"B" MOVIES         A dimwitted shut-in becomes the toast of Washington, D.C. society in this comedy starring Peter Sellers                                                                      being there
"B" SHARP          A joke says that when you play country music this way, your wife, your dog & your car return                                                                                   backwards
                   From the Italian for "jest", it's a clown or a fool                                                                                                                            a buffoon
                   Gaborone is the capital of this southern African country                                                                                                                        botswana
                   Sepia & mahogany are tones of this color                                                          

The above questions have been deemed to be similar or identical to previous questions asked on the show. We can investigate further to verify if this is indeed the case. 

Taking for example the question with the answer of `bellerophon` in the above dataframe - we can check the full dataset for instances where bellerophon is also the correct answer. We expect to see very similar questions.

In [111]:
jeopardy[jeopardy["Clean Answer"] == "bellerophon"]

,Show Number,Air Date,Round,Category,Value,Question,Answer,Clean Question,Clean Answer,Clean Value,Answer in Question,Question Overlap
170603,3831,2001-04-09,Double Jeopardy!,"""B"" SHARP",$600,This hero rode Pegasus,Bellerophon,this hero rode pegasus,bellerophon,600,0.0,1.0
173010,5338,2007-11-21,Double Jeopardy!,MAKE NO MYTHTAKE,$2000,He fell off Pegasus to his death,Bellerophon,he fell off pegasus to his death,bellerophon,2000,0.0,1.0
163776,5405,2008-02-22,Double Jeopardy!,"""B"" SHARP",$1200,This hero rode Pegasus,Bellerophon,this hero rode pegasus,bellerophon,1200,0.0,1.0
204878,6018,2010-11-10,Jeopardy!,MYTHOLOGY,$1000,He tamed the winged horse Pegasus with a bridle given to him by Athena,Bellerophon,he tamed the winged horse pegasus with a bridle given to him by athena,bellerophon,1000,0.0,1.0


Indeed we do. According to the sample of questions, there have been a total of **four** questions related to the mythological hero Bellerophon. It seems there is merit for contestants improving their knowledge of mythological figures.

In [112]:
jeopardy[jeopardy["Clean Answer"] == "aberdeen"]

,Show Number,Air Date,Round,Category,Value,Question,Answer,Clean Question,Clean Answer,Clean Value,Answer in Question,Question Overlap
140796,1237,1990-01-09,Jeopardy!,SCOTLAND,$500,"Scotland's ""Granite City""; its name means ""mouth of the Dee"" River, which is where it's located",Aberdeen,scotlands granite city its name means mouth of the dee river which is where its located,aberdeen,500,0.0,0.666667
151717,3273,1998-11-25,Double Jeopardy!,THERE'S SOMETHING ABOUT MARYLAND,$200,"This U.S. Army ""Proving Ground"" for weapons testing occupies over 70,000 acres in Harford County",Aberdeen,this us army proving ground for weapons testing occupies over 70000 acres in harford county,aberdeen,200,0.0,0.857143
210575,3773,2001-01-17,Jeopardy!,"""A"" IN GEOGRAPHY",$500,In the 1970s this Scottish fishing port became the center of the North Sea oil industry,Aberdeen,in the 1970s this scottish fishing port became the center of the north sea oil industry,aberdeen,500,0.0,1.000000
168430,4397,2003-10-21,Jeopardy!,"TAKE THE ""A"" TRAIN","$1,000",Scotrail's high speed Turbostar trains run on routes from Edinburgh to Glasgow & to this city,Aberdeen,scotrails high speed turbostar trains run on routes from edinburgh to glasgow to this city,aberdeen,0,0.0,0.666667
66790,4883,2005-11-30,Double Jeopardy!,EUROPEAN CITIES,$2000,"Known as the ""Granite City"", its name is Scots for ""At the Mouth of the Dee"", the river on which it lies",Aberdeen,known as the granite city its name is scots for at the mouth of the dee the river on which it lies,aberdeen,2000,0.0,1.000000
49613,5563,2008-11-12,Double Jeopardy!,"""A"" IN GEOGRAPHY",$2000,This Scottish seaport lies between the rivers Dee & Don,Aberdeen,this scottish seaport lies between the rivers dee don,aberdeen,2000,0.0,1.000000
169531,5776,2009-10-26,Double Jeopardy!,"""EEN""",$2000,"Scotland's third-largest city, it's known as the oil capital of Europe",Aberdeen,scotlands thirdlargest city its known as the oil capital of europe,aberdeen,2000,0.0,1.000000


`Aberdeen` has been the correct answer for **seven** questions in the dataset. Six of these seven occurrences have been related to the Scottish city. It is clear geographical knowledge is an important area of study, and this should not just be limited to US geography.

In [113]:
jeopardy[jeopardy["Clean Answer"] == "richard avedon"]

,Show Number,Air Date,Round,Category,Value,Question,Answer,Clean Question,Clean Answer,Clean Value,Answer in Question,Question Overlap
176983,2998,1997-09-17,Jeopardy!,"""A"" MEN",$500,This photographer known for his celebrity portraits learned his craft while in the Merchant Marine,Richard Avedon,this photographer known for his celebrity portraits learned his craft while in the merchant marine,richard avedon,500,0.0,1.0
210717,3593,2000-03-29,Double Jeopardy!,PHOTOGRAPHERS,$1000,"Truman Capote wrote the text for this Harper's Bazaar fashion photographer's 1959 collection ""Observances""",Richard Avedon,truman capote wrote the text for this harpers bazaar fashion photographers 1959 collection observances,richard avedon,1000,0.0,1.0
71255,4366,2003-09-08,Jeopardy!,PHOTOGRAPHERS,$600,"Dick Avery, Fred Astaire's character in ""Funny Face"", is based on this real-life photographer",Richard Avedon,dick avery fred astaires character in funny face is based on this reallife photographer,richard avedon,600,0.0,1.0
151383,4717,2005-02-22,Jeopardy!,"""A"" MEN",$600,This famed fashion photographer passed away in October 2004,(Richard) Avedon,this famed fashion photographer passed away in october 2004,richard avedon,600,0.0,1.0
6601,4985,2006-04-21,Jeopardy!,RICHARD,$800,He shot the famous photo of Nastassja Kinski & the serpent,Richard Avedon,he shot the famous photo of nastassja kinski the serpent,richard avedon,800,0.0,1.0
94950,5578,2008-12-03,Jeopardy!,PHOTOGRAPHY,$1000,"Truman Capote wrote the text for this fashion photographer's 1959 collection ""Observations""",Richard Avedon,truman capote wrote the text for this fashion photographers 1959 collection observations,richard avedon,1000,0.0,1.0


There have been six instances of questions related to the photographer `Richard Avedon` - there does indeed seem to be a trend of reoccurring questions, particularly in the context of historical people and figures (both from real life and fiction).

In [114]:
jeopardy[jeopardy["Clean Answer"] == "edward albee"]

,Show Number,Air Date,Round,Category,Value,Question,Answer,Clean Question,Clean Answer,Clean Value,Answer in Question,Question Overlap
92082,1197,1989-11-14,Double Jeopardy!,PLAYWRIGHTS,$400,"His first two produced plays were ""The Zoo Story"" and ""The Death of Bessie Smith""",Edward Albee,his first two produced plays were the zoo story and the death of bessie smith,edward albee,400,0.0,0.500000
54438,1255,1990-02-02,Double Jeopardy!,PLAYS,$600,"A very short-run play in 1963, written by D. Starkweather, was titled ""So Who's Afraid Of"" this playwright",Edward Albee,a very shortrun play in 1963 written by d starkweather was titled so whos afraid of this playwright,edward albee,600,0.0,0.666667
7932,1274,1990-03-01,Double Jeopardy!,AMERICAN PLAYS,$800,"This playwright dedicated ""A Delicate Balance"" to J. Steinbeck with ""affection and admiration""",Edward Albee,this playwright dedicated a delicate balance to j steinbeck with affection and admiration,edward albee,800,0.0,0.714286
27386,2339,1994-11-03,Final Jeopardy!,PLAYWRIGHTS,NaN,"He's won 3 Pulitzer Prizes for drama--in 1967, 1975 & 1994",Edward Albee,hes won 3 pulitzer prizes for dramain 1967 1975 1994,edward albee,0,0.0,0.666667
31634,2576,1995-11-13,Double Jeopardy!,PLAYS & PLAYWRIGHTS,$800,"The women in his play ""Three Tall Women"" are known by the letters ""A"", ""B"" & ""C"", not by names",Edward Albee,the women in his play three tall women are known by the letters a b c not by names,edward albee,800,0.0,1.000000
201075,2829,1996-12-12,Double Jeopardy!,PLAYWRIGHTS,$800,"Although his 1975 play ""Seascape"" had only a brief Broadway run, it won the Pulitzer Prize for Drama",Edward Albee,although his 1975 play seascape had only a brief broadway run it won the pulitzer prize for drama,edward albee,800,0.0,1.000000
176978,2998,1997-09-17,Jeopardy!,"""A"" MEN",$400,"In 1994 ""Three Tall Women"" earned this ""Seascape"" playwright his third Pulitzer Prize",Edward Albee,in 1994 three tall women earned this seascape playwright his third pulitzer prize,edward albee,400,0.0,1.000000
59703,3389,1999-05-06,Double Jeopardy!,NAME THE PLAYWRIGHT,$400,"""Who's Afraid of Virginia Woolf?""",Edward Albee,whos afraid of virginia woolf,edward albee,400,0.0,1.000000
77016,3490,1999-11-05,Jeopardy!,SCHOOL PLAYS,$400,"Some time passes before Jerry tells what happened at the zoo in this playwright's ""The Zoo Story""",Edward Albee,some time passes before jerry tells what happened at the zoo in this playwrights the zoo story,edward albee,400,0.0,1.000000
193209,4082,2002-05-07,Double Jeopardy!,PLAYBILL,$2000,"This playwright said that he's ""testing the limits of tolerance"" with his new play about 4 people & a goat",Edward Albee,this playwright said that hes testing the limits of tolerance with his new play about 4 people a goat,edward albee,2000,0.0,1.000000


Similarly, the American Playwright `Edward Albee` has been featured in 15 questions. There does indeed seem to be a pattern. It is certainly important for contestants to take note of which famous people and figures have appeared in past episodes as it is not out of the question that these questions will be recycled or slightly modified.

---
We could continue on looking into questions flagged to be similar, but to keep things concise, we can summarise the main key points:

* We estimate that around 1 in 20 questions have been asked before in some form or another.

* Looking into similar questions, it seems questions related to historical figures, people and geography are more frequently recycled. It would be worthwhile for contestants to take note of which famous people, countries and places appear as answers to questions as it is relatively likely that these will be answers to future questions in these categories.

## Low Value vs. High Value Questions

Here, we plan to determine if there is a relationship between terms used in questions and the question's monetary value. The `Chi-square` test can be used to answer this question.

To implement a Chi-square test:

* Questions can be classified as low-value (less than $800) or high-value (greater than $800)

* Looping through a list of all terms used: 

  - Find the number of low value questions each term occurs in.
  - Find the number of high value questions each term occurs in.
  - Find the percentage of questions the term occurs in.
  - Based on the percentage, find the expected count of questions the term occurs in.
  - Compute the Chi-square value from the observed and expected counts for high and low questions.

In [ ]:
def determine_value(row):
  
  """Classifies each question as either a high value question or a low value question. 1 if high value, 0 if low value."""
  
  if row["Clean Value"] > 800:
    return 1
  
  else:
    return 0
  
jeopardy["High Value"] = jeopardy.apply(determine_value, axis=1)

jeopardy.head()

,Show Number,Air Date,Round,Category,Value,Question,Answer,Clean Question,Clean Answer,Clean Value,Answer in Question,Question Overlap,High Value
84523,1,1984-09-10,Jeopardy!,LAKES & RIVERS,$100,River mentioned most often in the Bible,the Jordan,river mentioned most often in the bible,the jordan,100,0.000000,0.0,0
84565,1,1984-09-10,Double Jeopardy!,THE BIBLE,$1000,"According to 1st Timothy, it is the ""root of all evil""",the love of money,according to 1st timothy it is the root of all evil,the love of money,1000,0.333333,0.0,1
84566,1,1984-09-10,Double Jeopardy!,'50'S TV,$1000,Name under which experimenter Don Herbert taught viewers all about science,Mr. Wizard,name under which experimenter don herbert taught viewers all about science,mr wizard,1000,0.000000,0.0,1
84567,1,1984-09-10,Double Jeopardy!,NATIONAL LANDMARKS,$1000,D.C. building shaken by November '83 bomb blast,the Capitol,dc building shaken by november 83 bomb blast,the capitol,1000,0.000000,0.0,1
84568,1,1984-09-10,Double Jeopardy!,NOTORIOUS,$1000,"After the deed, he leaped to the stage shouting ""Sic semper tyrannis""",John Wilkes Booth,after the deed he leaped to the stage shouting sic semper tyrannis,john wilkes booth,1000,0.000000,0.0,1


In [140]:
jeopardy["High Value"].value_counts()

High Value
0    163901
1     53029
Name: count, dtype: int64

In this dataset, **just under 25%** of questions are considered high value questions.

In [ ]:
high_low_word_counts = {}

for i, row in jeopardy.iterrows():
    split_question = row["Clean Question"].split(" ")
    split_question = [word for word in split_question if len(word) > 5]  # remove words that are less than 6 letters long - this removes common stopwords such as "the", "than", "and", etc.
    high_value = row["High Value"]
    
    for word in split_question:
        if word not in high_low_word_counts:
            high_low_word_counts[word] = {'high': 0, 'low': 0}
        
        if high_value == 1:
            high_low_word_counts[word]['high'] += 1
        else:
            high_low_word_counts[word]['low'] += 1

A dictionary `high_low_word_counts` is created to store the number of times a given word appears in high and low valued questions. Below, the dictionary is converted to a dataframe.

In [117]:
high_low_word_counts = pd.DataFrame(high_low_word_counts)

high_low_word_counts.head()

,mentioned,according,timothy,experimenter,herbert,taught,viewers,science,building,shaken,november,leaped,shouting,semper,tyrannis,...,welfordon,bidfordon,coumerthe,ferberizing,flemenco,shalwar,seaquarium,knicker,lookinland,ingenhousz,magnetize,hrefhttpwwwjarchivecommedia20120127_j_22wmvherea,lagrange,standardizing,extralarge
high,39,97,13,3,25,49,12,98,146,4,84,3,3,6,2,...,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0
low,171,477,27,0,58,133,43,238,539,12,314,9,9,15,4,...,1,1,1,1,0,0,1,1,1,1,1,1,1,1,1


In [ ]:
def count_usage(word):
    """A helper function that returns the high and low counts for a given word using the high_low_word_counts lookup table."""
  
    if word in high_low_word_counts:
        data = high_low_word_counts[word]
        return data['high'], data['low']
    return 0, 0

Below, a list of all terms used (single words that are at least 6 characters long) in questions is assigned to the variable `terms_used_list`. This will be iterated over to create the `observed_counts` list which contains the observed frequencies of each word in the `terms_used_list` as tuples. This can be fed into the chi-square function to perform the statistical test.

In [119]:
terms_used_list = list(high_low_word_counts.columns)

observed_counts = []

for word in terms_used_list:
  observed_counts.append(count_usage(word))
  
observed_counts[:10]

[(np.int64(39), np.int64(171)),
 (np.int64(97), np.int64(477)),
 (np.int64(13), np.int64(27)),
 (np.int64(3), np.int64(0)),
 (np.int64(25), np.int64(58)),
 (np.int64(49), np.int64(133)),
 (np.int64(12), np.int64(43)),
 (np.int64(98), np.int64(238)),
 (np.int64(146), np.int64(539)),
 (np.int64(4), np.int64(12))]

To demonstrate the meaning of the `observed_counts` list, the observed counts for the first ten terms are shown above. The first element of this list is (39, 171) which denotes a term that appears 39 times in high value questions and 171 times in low value questions. 

Since order is preserved, we know this must correspond to the first element of the `terms_used_list` which is the word `"mentioned"`.

---
Finally, we move to performing the Chi-square test using the elements in the `observed_counts` list defined above. 

In order to conduct a chi-square test, expected or theoretical values must be found:

* The expected values are calculated by finding the proportion of questions containing the given term and multiplying this by the total number of high value questions and total number of low value questions (to find the expected high value count and expected low value count, respectively).

The chi-square value for each word can then be found:

$$
\chi^{2} = \displaystyle\sum_{i=1}^i\frac{(O_{i}-E_{i})^{2}}{E_{i}}
$$

>Additionally, in order for the statistical test to be reliable, the number of observations for a word must be sufficiently large (ideally, both frequencies should be greater than five). To ensure reliable results, any words which do not have at least five observations in both high or low value questions will not be tested. **Only** high frequency words are to be tested.

In [120]:
from scipy.stats import chisquare

high_value_count = jeopardy["High Value"].sum()
low_value_count = len(jeopardy) - high_value_count

chi_square_pvalues = []
index = 0                                       # initialize index variable to store the index of each word in the terms_used_list

for obs in observed_counts:
  
  if ((obs[0] < 5) or (obs[1] < 5)):            # for words with too few observations, do not perform chi-square test as the test is unreliable
    index += 1
    continue
  
  total = sum(obs)
  total_prop = total / jeopardy.shape[0]
  high_value_expected = total_prop * high_value_count
  low_value_expected = total_prop * low_value_count
  
  observed = np.array([obs[0], obs[1]])
  expected = np.array([high_value_expected, low_value_expected])
  p_value = chisquare(observed, expected)[-1]
  
  chi_square_pvalues.append((index, p_value))                      # store the index and calculated p-value for each word together as a tuple
  
  index += 1

chi_square_pvalues = pd.DataFrame(chi_square_pvalues, columns=["word_index", "p_value"])

In [139]:
high_value_count

np.int64(53029)

In [121]:
significant_words = chi_square_pvalues[chi_square_pvalues["p_value"] < 0.05]   # filter words with p-value less than 0.05
significant_words

,word_index,p_value
0,0,0.047635
1,1,0.000026
6,7,0.044030
10,15,0.000445
14,19,0.035167
...,...,...
7778,75368,0.009231
7779,77443,0.000073
7783,86253,0.001858
7784,88363,0.006304


The Chi-square test is conducted as described above. **1,191 words** have been identified to be statistically significant. This means for these words, there is sufficient evidence at the 5% significance level to claim that there is a relationship between the occurrence of these words in questions and the monetary value of the question.

In [122]:
print(f"{len(significant_words) / len(chi_square_pvalues) * 100:.2f}% of the high frequency words have a p-value less than 0.05")

15.29% of the high frequency words have a p-value less than 0.05


Of the high frequency words that were tested with the Chi-square test, just over 15% of the words were found to deviate significantly enough from the expected frequencies to be considered to be related to the monetary value of a question. 

This is not a particularly substantial number of words, but it could uncover some terms that occur more often in high or low value questions.

---
We can look into the words that were deemed to be related to the value of a question. The code block below uses the index values associated with the word and its corresponding observed frequencies to present a table containing the significant words alongside their associated p-values and observed frequency counts.

In [123]:
words = []
observed_count = []

for i, row in significant_words.iterrows():
  
  word_index = int(row["word_index"])
  
  word = terms_used_list[word_index]
  
  words.append(word)
  
  observed = observed_counts[word_index]
  
  observed_count.append(observed)
  
significant_words["word"] = words
significant_words["observed_count"] = observed_count

In [ ]:
significant_words.set_index("word_index", inplace=True)   # set word_index as the index for presentation clarity

In [130]:
significant_words.sort_values("p_value").head(50)

,p_value,word,observed_count
word_index,,,
5028,4.792761e-83,target_blankthisa,"(731, 892)"
6520,1.045860e-66,target_blankherea,"(972, 1497)"
65777,7.059866e-48,monitora,"(239, 202)"
45465,1.173842e-45,target_blankjimmy,"(323, 351)"
50491,9.789275e-36,target_blanksarah,"(323, 406)"
9269,1.319465e-35,reports,"(490, 745)"
65485,1.068178e-30,target_blankkelly,"(229, 260)"
65468,2.456252e-25,target_blankjon,"(171, 184)"
50519,1.376021e-24,target_blankcheryl,"(190, 222)"


## Remarks on the Chi-square Test Results

Firstly, it must be stated that of the 99,975 unique words featured in questions (which are six or more characters long), **only** 1,191 of these words (1.2%) could be shown to be statistically different regarding their appearance in high vs. low value questions. This is a relatively **small** proportion, so it does not seem to be hugely important. Nevertheless, some limited insight can still be gained.

Before looking into specific words, it is important to contextualize the results:

In the full dataset of questions, just under 25% of the questions were found to be high value (greater than $800). Taking our null hypothesis to be true for every unqiue word in the `terms_used_list`, this would mean we would expect the number of times each word appears in a high value vs. low value question to be in the ratio of 1:3. For instance, take the word `"battle"` which occurred 903 times in questions. If our null hypothesis is true, we would expect the number of appearances of the word across high and low value questions to roughly be in the ratio 1:3 (226, 677). Instead, the observed counts for the word `"battle"` is (282, 621). This suggests the word occurs disproportionately **more** in higher value questions. 

This may suggest it is worth prioritising knowledge of historical battles to improve the likelihood that a contestant wins more prize money.

---

Now we can look into some other specific words and make additional observations:

* Words starting with `target_` correspond to questions that contain embedded links (meaning the question either required the contestant to view an image or listen to an audio clue). The occurrences of these words across the high vs. low value questions is certainly skewed more towards **high value** questions (around 1:2 rather than the expected 1:3 ratio). This could suggest visual and audio questions are more likely to be higher value questions.

* Interestingly, the `3named` word is heavily skewed towards higher value questions. This word occurs only 32 times in questions. If we take our null hypothesis to be true (1:3 ratio), we would expect the apperances of "3-named" to be (8, 24) - 8 in high value and 24 in low value questions. However, we observe (25, 7). This would suggest questions about famous people with three names are highly likely to be high value questions.

We could continue on, but for the sake of time, we can stop here.

In [141]:
jeopardy[jeopardy["Clean Question"].str.contains("target_blankseena")].sample(5) # example of a question containing the word "target_blankseena"

,Show Number,Air Date,Round,Category,Value,Question,Answer,Clean Question,Clean Answer,Clean Value,Answer in Question,Question Overlap,High Value
9652,5530,2008-09-26,Jeopardy!,WHAT TO WEAR,$1000,"You might see celebrities Don or Beth in this style of shirt <a href=""http://www.j-archive.com/media/2008-09-26_J_24.jpg"" target=""_blank"">seen</a> <a href=""http://www.j-archive.com/media/2008-09-26_J_24a.jpg"" target=""_blank"">here</a>",a Henley style,you might see celebrities don or beth in this style of shirt a hrefhttpwwwjarchivecommedia20080926_j_24jpg target_blankseena a hrefhttpwwwjarchivecommedia20080926_j_24ajpg target_blankherea,a henley style,1000,0.5,0.600000,1
140490,6241,2011-11-07,Jeopardy!,ON A U.S. POSTAGE STAMP,$800,"The dinnerware <a href=""http://www.j-archive.com/media/2011-11-07_J_23.jpg"" target=""_blank"">seen</a> <a href=""http://www.j-archive.com/media/2011-11-07_J_23a.jpg"" target=""_blank"">here</a>, with this Spanish name",Fiesta ware,the dinnerware a hrefhttpwwwjarchivecommedia20111107_j_23jpg target_blankseena a hrefhttpwwwjarchivecommedia20111107_j_23ajpg target_blankherea with this spanish name,fiesta ware,800,0.0,0.666667,0
49168,5077,2006-10-10,Double Jeopardy!,"WHAT THE ""H""?",$400,"The 17th-century work <a href=""http://www.j-archive.com/media/2006-10-10_DJ_01.jpg"" target=""_blank"">seen</a> <a href=""http://www.j-archive.com/media/2006-10-10_DJ_01a.jpg"" target=""_blank"">here</a> shows a landscape in this country",Holland,the 17thcentury work a hrefhttpwwwjarchivecommedia20061010_dj_01jpg target_blankseena a hrefhttpwwwjarchivecommedia20061010_dj_01ajpg target_blankherea shows a landscape in this country,holland,400,0.0,0.714286,0
206499,5814,2009-12-17,Double Jeopardy!,SCIENCE,$1600,"A depiction of this dinosaur of the Cretaceous period is <a href=""http://www.j-archive.com/media/2009-12-17_DJ_25.jpg"" target=""_blank"">seen</a> <a href=""http://www.j-archive.com/media/2009-12-17_DJ_25a.jpg"" target=""_blank"">here</a>",a Triceratops,a depiction of this dinosaur of the cretaceous period is a hrefhttpwwwjarchivecommedia20091217_dj_25jpg target_blankseena a hrefhttpwwwjarchivecommedia20091217_dj_25ajpg target_blankherea,a triceratops,1600,0.0,0.750000,1
48411,4768,2005-05-04,Double Jeopardy!,AROUND THE LOUVRE,$2000,"Since the 1600s the Louvre has owned the work <a href=""http://www.j-archive.com/media/2005-05-04_DJ_11.jpg"" target=""_blank"">seen</a> <a href=""http://www.j-archive.com/media/2005-05-04_DJ_11a.jpg"" target=""_blank"">here</a> by Claude Gellée, also known as this, for his region of birth",Claude Lorrain,since the 1600s the louvre has owned the work a hrefhttpwwwjarchivecommedia20050504_dj_11jpg target_blankseena a hrefhttpwwwjarchivecommedia20050504_dj_11ajpg target_blankherea by claude gellée also known as this for his region of birth,claude lorrain,2000,0.5,0.625000,1


In [142]:
jeopardy[jeopardy["Clean Question"].str.contains("3named")].sample(10)

,Show Number,Air Date,Round,Category,Value,Question,Answer,Clean Question,Clean Answer,Clean Value,Answer in Question,Question Overlap,High Value
214168,6244,2011-11-10,Double Jeopardy!,ECONOMIST DANCE PARTY,$1200,"This 3-named Brit's ""General Theory"" says unemployment is reduced with more government spending; how low can you go?",John Maynard Keynes,this 3named brits general theory says unemployment is reduced with more government spending how low can you go,john maynard keynes,1200,0.0,1.000000,1
155515,5265,2007-06-29,Double Jeopardy!,JOHN LEGEND,$2000,"This 3-named American artist was known for portraits of socially prominent people, like the women seen <a href=""http://www.j-archive.com/media/2007-06-29_DJ_15.jpg"" target=""_blank"">here</a>",John Singer Sargent,this 3named american artist was known for portraits of socially prominent people like the women seen a hrefhttpwwwjarchivecommedia20070629_dj_15jpg target_blankherea,john singer sargent,2000,0.0,0.888889,1
14105,5110,2006-11-24,Jeopardy!,THE ONE I LOVE,$1000,"In 1925 this 3-named British economist married Lydia Lopakova, one of Diaghilev's ballerinas",John Maynard Keynes,in 1925 this 3named british economist married lydia lopakova one of diaghilevs ballerinas,john maynard keynes,1000,0.0,0.857143,1
47813,4365,2003-07-18,Double Jeopardy!,PART: CHARLES,"$1,000",3-named actor who played 3-named TV surgeon Charles Emerson Winchester,David Ogden Stiers,3named actor who played 3named tv surgeon charles emerson winchester,david ogden stiers,0,0.0,1.000000,0
46947,5227,2007-05-08,Jeopardy!,NAME THAT TUNE-STER,$1000,"A 3-named woman: ""Like A Star"", ""Enchantment"", ""Put Your Records On""",Corinne Bailey Rae,a 3named woman like a star enchantment put your records on,corinne bailey rae,1000,0.0,1.000000,1
215134,6036,2010-12-06,Double Jeopardy!,PAINTERS,$2000,The influence of Las Meninas is apparent in the Daughters of Edward Darley Boit by this 3-named artist,John Singer Sargent,the influence of las meninas is apparent in the daughters of edward darley boit by this 3named artist,john singer sargent,2000,0.0,1.000000,1
66968,4893,2005-12-14,Double Jeopardy!,THE VATICAN'S INDEX OF FORBIDDEN BOOKS,$2000,"""Madame Bovary"" by Flaubert, we understand, but ""Principles of Political Economy"" by this 3-named Brit, we don't",John Stuart Mill,madame bovary by flaubert we understand but principles of political economy by this 3named brit we dont,john stuart mill,2000,0.0,1.000000,1
156681,5432,2008-04-01,Jeopardy!,COLD,$800,Charlotte Pass in this 3-named state holds Australia's cold-temperature record with -9° F. in 1994,New South Wales,charlotte pass in this 3named state holds australias coldtemperature record with 9 f in 1994,new south wales,800,0.0,0.800000,0
207517,5791,2009-11-16,Double Jeopardy!,THEY SHOT THE SHERIFF,$2000,"On his 21st birthday, this 3-named gunman shot & killed Deputy Charles Webb in Brown County, Texas",John Wesley Hardin,on his 21st birthday this 3named gunman shot killed deputy charles webb in brown county texas,john wesley hardin,2000,0.0,1.000000,1
136716,5909,2010-04-29,Double Jeopardy!,I DETECT A DETECTIVE,$1600,No objection--this 3-named author created the dogged defense lawyer Perry Mason,Erle Stanley Gardner,no objectionthis 3named author created the dogged defense lawyer perry mason,erle stanley gardner,1600,0.0,0.857143,1


## Final Thoughts

A number of insights into patterns of Jeopardy! questions have been uncovered through this analysis.

* Around 9% of the answers in the dataset have words featured in the question. 

  - There are some categories of question such as `HIDDEN COUNTRIES` or `THE LARGEST IN AREA` where the answer is certain to be in the question - either split between two or more words as is the case for `HIDDEN COUNTRIES` or listed as possible answers as is the case for `THE LARGEST IN AREA`. There is a strong geography theme for these questions, meaning it is important for contestants to be knowledgable with world geography.

  - The `STUPID ANSWERS` category questions often contain the answer in plain sight. It is certainly worthwhile for contestants to practice these types of question.

* We can approximate **just under 1 in 20 questions have been asked previously in some form or another**.

  - Analysis of similar questions, it seems questions related to historical figures, people and geography are more frequently recycled. It would be particularly worthwhile for contestants to take note of which famous people and places appear as answers to questions as it is relatively likely that these will be answers to future questions in these categories.

* Of the 99,975 6+ long unique words featured in questions, 1,191 of these words (1.2%) could be shown to be statistically different regarding their appearance in high vs. low value questions.

  - Although this is not a huge proportion of the words, this did lead to some interesting observations explained in the above chi-square section.

It should not be understated that this is a **challenging** quiz show - contestants will need broad general knowledge to perform well. Focus on preparing for the show by studying up on **general knowledge** - with a particular emphasis on US history, US popular culture and wider geographical knowledge. This will stand competitors in good stead.